# Track A2 — Kaggle Synthetic Fine-Tune Demo

---

> ## ⚠️ SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE — NOT OFFICIAL LEGAL TEXT
>
> This notebook is for **AI engineering portfolio demonstration only**.
>
> - Uses **only synthetic-demo data** (no real legal corpus).
> - Does **NOT** use `UTS_VLC` candidates.
> - Does **NOT** use `corpus_candidate_manifest` as training data.
> - Does **NOT** prove legal correctness.
> - All outputs are **synthetic-demo artifacts only**.
> - Model output is **NOT legal advice** and must not be used for actual legal compliance.
> - Track B real legal corpus governance remains **blocked** for QA/SFT/RAG.

---

## What this notebook demonstrates

1. Dataset loading from a Kaggle-uploaded synthetic-demo dataset.
2. Dataset validation before training (synthetic labels, disclaimers, field constraints).
3. Prompt formatting from Alpaca-style JSONL.
4. LoRA/QLoRA-style fine-tune setup for the flagship 7B Kaggle target.
5. Fine-tuning on synthetic data for one epoch (Kaggle GPU only).
6. Evaluation comparing base vs. adapter on synthetic test examples.
7. Benchmark export to `/kaggle/working/track_a_benchmark_results.json`.
8. Screenshot guidance for portfolio evidence.

**Portfolio context:** This is Track A of the ViLegal-Agent project — the synthetic AI engineering demo track. It is intentionally separated from Track B (real legal corpus governance), which remains safety-gated.

## Section 1 — Configuration

Configure paths, model name, and training hyperparameters here.
All settings are for Kaggle execution only.

In [ ]:
# ============================================================
# Track A2 — Kaggle Synthetic Fine-Tune Demo
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE — NOT OFFICIAL LEGAL TEXT
# ============================================================

import os
import json
from pathlib import Path

# -----------------------------------------------------------
# Kaggle dataset input path (after uploading the packaged dataset)
# -----------------------------------------------------------
KAGGLE_INPUT_DIR = Path("/kaggle/input/vilegal-synthetic-demo")
TRAIN_FILE = KAGGLE_INPUT_DIR / "train.jsonl"
VALIDATION_FILE = KAGGLE_INPUT_DIR / "validation.jsonl"
TEST_FILE = KAGGLE_INPUT_DIR / "test.jsonl"

# -----------------------------------------------------------
# Model configuration
# 7B is the flagship Kaggle target for the portfolio demo.
# 3B is the local/dev baseline because the user can already run it locally.
# 0.5B is smoke-test only and not the portfolio target.
# -----------------------------------------------------------
BASE_MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"
# Local/dev fallback: "Qwen/Qwen2.5-3B-Instruct"
# Smoke-test only fallback: "Qwen/Qwen2.5-0.5B-Instruct"

# -----------------------------------------------------------
# Training configuration
# -----------------------------------------------------------
OUTPUT_DIR = "/kaggle/working/vilegal-synthetic-demo-adapter"
NUM_TRAIN_EPOCHS = 1
MAX_STEPS = None          # Set to an integer (e.g. 100) to limit steps for quick demo
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQ_LENGTH = 1024
USE_QLORA = True          # Use 4-bit quantization if bitsandbytes is available

# -----------------------------------------------------------
# Evaluation configuration
# -----------------------------------------------------------
EVAL_SAMPLE_SIZE = 50     # Number of test examples to evaluate
BENCHMARK_OUTPUT = "/kaggle/working/track_a_benchmark_results.json"

# -----------------------------------------------------------
# Safety constants — DO NOT CHANGE
# -----------------------------------------------------------
REQUIRED_SOURCE = "synthetic-demo"
REQUIRED_IS_SYNTHETIC = True
REQUIRED_IS_LEGAL_GROUND_TRUTH = False
REQUIRED_APPROVED_FOR_RAG_INDEX = False

DISCLAIMER = "Synthetic demo only. Not legal advice. Not official legal text. Not legal-ground-truth."

print("Configuration loaded.")
print(f"  BASE_MODEL_NAME: {BASE_MODEL_NAME}")
print(f"  OUTPUT_DIR:      {OUTPUT_DIR}")
print(f"  DISCLAIMER:      {DISCLAIMER}")

## Section 2 — Dataset Validation

Validate the synthetic dataset **before** training to confirm all safety constraints.
This cell will raise an error and halt execution if any row fails validation.

In [ ]:
# ============================================================
# Dataset Validation
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================

def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    rows = []
    with open(path, "r", encoding="utf-8") as fh:
        for i, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {i} of {path}: {exc}") from exc
    return rows


def validate_split(rows, split_name, expected_count):
    """Validate all rows in a split for synthetic-only safety constraints."""
    errors = []
    for i, row in enumerate(rows, start=1):
        if row.get("source") != REQUIRED_SOURCE:
            errors.append(f"Row {i}: source={row.get('source')!r} (expected 'synthetic-demo')")
        if row.get("is_synthetic") is not REQUIRED_IS_SYNTHETIC:
            errors.append(f"Row {i}: is_synthetic={row.get('is_synthetic')!r} (expected true)")
        if row.get("is_legal_ground_truth") is not REQUIRED_IS_LEGAL_GROUND_TRUTH:
            errors.append(f"Row {i}: is_legal_ground_truth must be false")
        if row.get("approved_for_rag_index") is not REQUIRED_APPROVED_FOR_RAG_INDEX:
            errors.append(f"Row {i}: approved_for_rag_index must be false")
    
    if errors:
        raise ValueError(
            f"[VALIDATION FAIL] {split_name} — {len(errors)} safety violations:\n"
            + "\n".join(errors[:20])
        )
    
    if len(rows) != expected_count:
        raise ValueError(
            f"[VALIDATION FAIL] {split_name}: expected {expected_count} rows, got {len(rows)}."
        )
    
    print(f"  [OK] {split_name}: {len(rows)} rows — all safety constraints passed.")


print("=" * 60)
print("Dataset Validation")
print("SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE")
print("=" * 60)

train_rows = load_jsonl(TRAIN_FILE)
val_rows = load_jsonl(VALIDATION_FILE)
test_rows = load_jsonl(TEST_FILE)

validate_split(train_rows, "train", 2000)
validate_split(val_rows, "validation", 250)
validate_split(test_rows, "test", 250)

print("\nAll splits validated. Proceeding to training.")
print(f"DISCLAIMER: {DISCLAIMER}")

## Section 3 — Data Formatting

Format Alpaca-style JSONL rows into prompt/completion text strings
suitable for causal language model fine-tuning.

In [ ]:
# ============================================================
# Alpaca-style prompt/completion formatting
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================

def format_prompt(row):
    """Format a single Alpaca-style row as a training prompt string."""
    instruction = row.get("instruction", "").strip()
    context = row.get("input", "").strip()
    output = row.get("output", "").strip()
    disclaimer = row.get("disclaimer", DISCLAIMER)
    
    if context:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Context\n{context}\n\n"
            f"### Response\n{output}"
        )
    else:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Response\n{output}"
        )
    return prompt_text


def format_inference_prompt(row):
    """Format a row for inference (no output label)."""
    instruction = row.get("instruction", "").strip()
    context = row.get("input", "").strip()
    disclaimer = row.get("disclaimer", DISCLAIMER)
    
    if context:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Context\n{context}\n\n"
            f"### Response\n"
        )
    else:
        prompt_text = (
            f"### Disclaimer\n{disclaimer}\n\n"
            f"### Instruction\n{instruction}\n\n"
            f"### Response\n"
        )
    return prompt_text


# Quick sanity check on formatting
sample_row = train_rows[0]
formatted_sample = format_prompt(sample_row)

print("Sample formatted training example:")
print("-" * 60)
print(formatted_sample[:800])
print("-" * 60)
print(f"\nTotal training examples: {len(train_rows)}")
print(f"Total validation examples: {len(val_rows)}")
print(f"Total test examples: {len(test_rows)}")

## Section 4 — LoRA / QLoRA Setup

Load the flagship Kaggle base model and configure LoRA adapters.
This cell runs only on Kaggle with GPU access.

> **Model profile note:** 7B is the flagship Kaggle target. 3B is the local/dev baseline. 0.5B is smoke-test only.
> **Note:** This is a synthetic fine-tune demo. The model output is NOT legal advice.
> Track B real legal corpus training remains blocked.

In [ ]:
# ============================================================
# LoRA/QLoRA Setup — Kaggle Only
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print(f"Loading tokenizer: {BASE_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Configure quantization for QLoRA if available
bnb_config = None
if USE_QLORA:
    try:
        import bitsandbytes as bnb  # noqa: F401
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        print("QLoRA (4-bit) quantization configured.")
    except ImportError:
        print("bitsandbytes not available — using standard LoRA (fp16/bf16).")

print(f"Loading model: {BASE_MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if bnb_config is None else None,
)

if bnb_config is not None:
    model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"\nDISCLAIMER: {DISCLAIMER}")

## Section 5 — Fine-Tune on Synthetic Data (Kaggle Only)

Train the LoRA adapter on the synthetic-demo training split using the 7B Kaggle target by default.

> **Reminder:** This demonstrates the ML fine-tune workflow mechanics using synthetic data only.
> It does NOT demonstrate legal correctness. Model output is NOT legal advice.

In [ ]:
# ============================================================
# Fine-Tune Cell — Kaggle Synthetic Training
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# Do NOT run locally. Intended for Kaggle GPU execution only.
# ============================================================

from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset

# Format all training rows
formatted_train = [{"text": format_prompt(row)} for row in train_rows]
formatted_val = [{"text": format_prompt(row)} for row in val_rows]

train_dataset = Dataset.from_list(formatted_train)
val_dataset = Dataset.from_list(formatted_val)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else -1,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=False,
    bf16=True,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",   # No W&B / MLflow — Kaggle demo only
    run_name="track_a2_synthetic_finetune_demo",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
)

print("=" * 60)
print("Starting synthetic fine-tuning...")
print(f"DISCLAIMER: {DISCLAIMER}")
print("=" * 60)

trainer.train()

# Save adapter only (not full model weights)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining complete.")
print(f"Adapter saved to: {OUTPUT_DIR}")
print(f"DISCLAIMER: {DISCLAIMER}")
# NOTE: push_to_hub is intentionally disabled.
# To share, set push_to_hub=True in TrainingArguments and provide a hub token.
# This is disabled by default as per Track A2 safety policy.

## Section 6 — Evaluation: Base vs Adapter on Synthetic Test Examples

Compare the base model and the fine-tuned adapter on a sample of synthetic test examples.

Metrics collected:
- `format_valid_rate`: fraction of responses containing expected format markers
- `disclaimer_present_rate`: fraction of prompts that included disclaimer in context
- `synthetic_task_pass_rate`: proxy metric based on keyword matching in response

> **Note:** These are proxy metrics on synthetic data only. They do NOT measure real legal accuracy.

In [ ]:
# ============================================================
# Evaluation: Base vs Adapter
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================

import random
from peft import PeftModel

random.seed(42)
eval_samples = random.sample(test_rows, min(EVAL_SAMPLE_SIZE, len(test_rows)))


def generate_response(model_obj, tokenizer_obj, prompt_str, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    inputs = tokenizer_obj(
        prompt_str,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH - max_new_tokens,
    ).to(model_obj.device)
    
    with torch.no_grad():
        outputs = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer_obj.pad_token_id,
        )
    
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer_obj.decode(generated, skip_special_tokens=True).strip()


def score_response(response_str, reference_output, task_type):
    """Compute proxy metrics. Not a real legal evaluation."""
    # Format valid: check for non-empty response
    format_valid = len(response_str.strip()) > 10
    
    # Synthetic task pass: simple keyword overlap proxy
    ref_words = set(reference_output.lower().split())
    resp_words = set(response_str.lower().split())
    overlap = len(ref_words & resp_words) / max(len(ref_words), 1)
    synthetic_pass = overlap >= 0.15  # Low threshold for proxy only
    
    return {"format_valid": format_valid, "synthetic_pass": synthetic_pass}


# Evaluate base model
print("Evaluating base model on synthetic test examples...")
base_results = []
for row in eval_samples:
    prompt = format_inference_prompt(row)
    response = generate_response(model, tokenizer, prompt)
    scores = score_response(response, row.get("output", ""), row.get("task_type", ""))
    base_results.append({"prompt": prompt, "response": response, **scores})

base_format_rate = sum(r["format_valid"] for r in base_results) / len(base_results)
base_pass_rate = sum(r["synthetic_pass"] for r in base_results) / len(base_results)
print(f"Base model  — format_valid_rate: {base_format_rate:.2%}, synthetic_task_pass_rate: {base_pass_rate:.2%}")

# Load and evaluate adapter
print("\nLoading adapter for evaluation...")
adapter_model = PeftModel.from_pretrained(model, OUTPUT_DIR)
adapter_model.eval()

print("Evaluating adapter model on synthetic test examples...")
adapter_results = []
for row in eval_samples:
    prompt = format_inference_prompt(row)
    response = generate_response(adapter_model, tokenizer, prompt)
    scores = score_response(response, row.get("output", ""), row.get("task_type", ""))
    adapter_results.append({"prompt": prompt, "response": response, **scores})

adapter_format_rate = sum(r["format_valid"] for r in adapter_results) / len(adapter_results)
adapter_pass_rate = sum(r["synthetic_pass"] for r in adapter_results) / len(adapter_results)
print(f"Adapter     — format_valid_rate: {adapter_format_rate:.2%}, synthetic_task_pass_rate: {adapter_pass_rate:.2%}")

print(f"\nDISCLAIMER: {DISCLAIMER}")

## Section 7 — Benchmark Export

Export results to `/kaggle/working/track_a_benchmark_results.json` for portfolio evidence.

In [ ]:
# ============================================================
# Benchmark Export
# SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE
# ============================================================

benchmark_report = {
    "track": "A",
    "phase": "A2",
    "name": "Kaggle Synthetic Fine-Tune Demo",
    "disclaimer": DISCLAIMER,
    "dataset": {
        "source": "synthetic-demo",
        "train_rows": len(train_rows),
        "validation_rows": len(val_rows),
        "test_rows": len(test_rows),
        "is_legal_ground_truth": False,
        "approved_for_rag_index": False,
    },
    "model": {
        "base_model": BASE_MODEL_NAME,
        "use_lora": True,
        "use_qlora": USE_QLORA,
        "push_to_hub": False,
    },
    "results": [
        {
            "model": "base",
            "dataset": "synthetic-demo/test",
            "examples": len(eval_samples),
            "format_valid_rate": round(base_format_rate, 4),
            "synthetic_task_pass_rate": round(base_pass_rate, 4),
            "disclaimer_present_rate": 1.0,
            "notes": "Base model before fine-tuning. Synthetic proxy metrics only.",
        },
        {
            "model": "adapter (LoRA)",
            "dataset": "synthetic-demo/test",
            "examples": len(eval_samples),
            "format_valid_rate": round(adapter_format_rate, 4),
            "synthetic_task_pass_rate": round(adapter_pass_rate, 4),
            "disclaimer_present_rate": 1.0,
            "notes": "LoRA adapter fine-tuned on synthetic-demo train split. Synthetic proxy metrics only.",
        },
    ],
    "safety_confirmation": {
        "use_real_legal_corpus": False,
        "use_uts_vlc_candidates": False,
        "use_corpus_candidate_manifest": False,
        "mark_as_legal_ground_truth": False,
        "approved_for_rag_index": False,
        "allow_local_training": False,
        "allow_hub_push_by_default": False,
        "track_b_unblocked": False,
    },
}

with open(BENCHMARK_OUTPUT, "w", encoding="utf-8") as fh:
    json.dump(benchmark_report, fh, indent=2, ensure_ascii=False)

print(f"Benchmark results exported to: {BENCHMARK_OUTPUT}")
print("\nBenchmark summary:")
for result in benchmark_report["results"]:
    print(f"  {result['model']:20s} | format_valid={result['format_valid_rate']:.2%} | synthetic_pass={result['synthetic_task_pass_rate']:.2%}")

print(f"\nDISCLAIMER: {DISCLAIMER}")

## Section 8 — Portfolio Evidence Checklist

Use this section to capture screenshots and notes for your portfolio.

### Screenshot Checklist

- [ ] **Dataset validation output** — shows synthetic-only constraints passing.
- [ ] **Training loss curve** — from the Kaggle training log.
- [ ] **Base vs adapter comparison table** — `format_valid_rate` and `synthetic_task_pass_rate`.
- [ ] **Benchmark JSON** — contents of `/kaggle/working/track_a_benchmark_results.json`.
- [ ] **Adapter output directory listing** — shows adapter files saved in `/kaggle/working/`.

### Portfolio Interpretation

| What this shows | What this does NOT show |
|---|---|
| ML fine-tune workflow mechanics | Real legal accuracy |
| LoRA/QLoRA setup and training | Legal correctness |
| Synthetic dataset pipeline | Ground-truth legal text |
| Benchmark comparison methodology | Production-ready legal AI |
| Fail-closed safety validation | Actual legal advice |

### Required Disclaimer for Any Public Use

> **SYNTHETIC DEMO ONLY — NOT LEGAL ADVICE — NOT OFFICIAL LEGAL TEXT**
> 
> Track A demonstrates AI engineering mechanics, not real legal capability.
> Track B real legal corpus governance remains blocked for QA/SFT/RAG.

---

**Next steps after collecting portfolio evidence:**
Download `track_a_benchmark_results.json` from the Kaggle working directory.
Proceed to Track A3 public portfolio packaging.